# 공개 코드 재현 — FinRL(EIIE·EI3)의 평가 시작 상태 효과

**목적.** 다른 연구자의 공개 구현을 **코드 수정 없이** 실행하고, 평가 시작 보유만 바꿨을 때 결과가 얼마나 달라지는지,
그 크기가 닫힌 형태 공식으로 사전에 예측되는지 검증한다.

**대상 코드.** FinRL 커밋 `7e98df4`의 `PortfolioOptimizationEnv`, `PolicyGradient`, `EIIE`(Jiang et al. 2017), `EI3`(Shi et al. 2019).
원본 파일을 실행 시점에 그대로 받아 쓴다. 바꾸는 것은 초기값 두 곳뿐이다 — 환경의 첫 보유 비중(`_actions_memory[0]`, `_final_weights[0]`)과
직전 행동 메모리(PVM) 초기값. 학습·평가 루프는 FinRL의 `train`/`test`를 그대로 호출하며, 매 설정마다 '현금 시작'이 수정하지 않은 원본 실행과
같은 값을 내는지 확인한다. GPM은 원 논문의 기업 관계 그래프가 코드와 함께 배포되지 않아 제외했다.

**설정(공식 데모 기준).** 일봉 Yahoo 데이터, 종목별 MaxAbs 정규화, 학습 2011-01-01~2019-12-30, 테스트 2020·2021·2022 각 1년,
TRF 수수료, time_window 50, 특징 close/high/low, lr 0.01, 40 에피소드, 테스트 중 온라인 학습(10스텝마다).

| 축 | 값 |
|---|---|
| 구조 | EIIE(데모 kwargs: k_size 3), EI3(기본 kwargs) |
| 시장 | 브라질 10종목(공식 데모 목록) / 미국 다우30(FinRL `DOW_30_TICKER`, 같은 커밋) |
| 수수료 | 0.25%(데모) / 0%(대조군 — 관성을 학습할 유인이 없음) |
| 시드 | 0, 1, 2 (데모에는 시드가 없음) |
| 데이터 규칙 | 2011-01-31 이후에 상장된 종목은 제외, 남은 종목이 모두 가격을 가진 날짜만 사용(제외 종목은 출력·기록) |

**측정.**
- 관성 추정(학습 구간만): 학습 구간 20개 시점에서 현금·등가중 시작으로 60거래일 실행(온라인 꺼짐) → 노출(1 − 현금 비중) 차이(시작 차이 1로 나눈 값) 평균의
  k = 0~59 누적 합이 Σ (1 − α)^(k+1) 과 같아지는 α를 α_eff로 한다(관성이 없으면 1, 클수록 작음 — 누적 편차가 같은 기하 감쇠의 α). x̄ = 같은 시점에서 60거래일 워밍업 뒤 평균 노출.
- 예측(테스트 실행 전 확정): 연도별 (x₀ − x̄) · Σ_k (1 − α_eff)^(k+1) · log(1 + b_k), b = 종목 등가중 일간 수익, x₀ = 0(현금)·1(등가중).
- 관측(온라인 꺼짐): log 성장(해당 시작) − log 성장(60거래일 워밍업 후).
- 데모 원래 방식(온라인 켜짐): 현금 시작·등가중 시작 각각의 log 성장, 데모 벤치마크(매 스텝 등가중 리밸런싱, 현금 시작)와의 차이.

**사전 등록 판정(실행 전 커밋).**

| 가설 | 기준 |
|---|---|
| H6 학습된 관성 | 수수료 0.25%에서 (구조 × 시장) 4개 칸 중 3개 이상이, 칸 안 시드 3개 중 2개 이상 α_eff < 0.5 |
| H6b 비용-관성 관계 | (구조 × 시장 × 시드) 12쌍 중 9쌍 이상에서 α_eff(0.25%) < α_eff(0%) |
| H7 공식 예측 | 모든 설정(24개)의 (연도 × 시작) 쌍 합산: 관측 vs 예측 상관 ≥ 0.8, 회귀 기울기 ∈ [0.7, 1.3]. 단 예측의 표준편차가 0.5 로그%p 미만이면(모두 관성이 없어 예측 범위가 좁음) '판정 불가'로 하고 관측 효과의 최대·95분위를 보고 |
| H8 보고 수익 영향 | 수수료 0.25%·온라인 켜짐(데모 방식)에서 (구조 × 시장) 칸마다 Δ = log 성장(등가중 시작) − log 성장(현금 시작)의 9개 값(시드 × 연도) \|Δ\| 중앙값 ≥ 1.0 로그%p 이면 '실질적 영향'. 4개 칸 중 3개 이상이면 전체 '실질적 영향'. 벤치마크 대비 초과수익 부호가 시작 보유에 따라 바뀐 셀 수도 보고 |

**실행.** `런타임 > 모두 실행`. CPU로 약 4~6시간(학습 24회). **끊기면 다시 `모두 실행`** — 학습은 에피소드마다 정책·옵티마이저·난수 상태를
Drive에 저장해 끊긴 지점부터 같은 결과로 이어지고, 평가는 설정별로 저장되어 끝난 설정은 건너뛴다.
결과: Drive `data/finrl_replication/` (`raw_*.csv`, `alpha.csv`, `windows.csv`, `protocol.csv`, `checks.csv`).


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q gym==0.26.2 torch_geometric quantstats yfinance stable-baselines3

In [ ]:
# FinRL 원본(커밋 7e98df4)을 수정 없이 받아 쓴다
import sys, urllib.request, pathlib, random, warnings, logging
COMMIT = '7e98df461cbc7772f16a545a2f13b521e5d0b8f4'
RAW = f'https://raw.githubusercontent.com/AI4Finance-Foundation/FinRL/{COMMIT}/finrl/'
PKG = pathlib.Path('/content/finrl_po'); PKG.mkdir(exist_ok=True); (PKG / '__init__.py').write_text('')
for f in ['agents/portfolio_optimization/algorithms.py', 'agents/portfolio_optimization/architectures.py',
          'agents/portfolio_optimization/utils.py', 'agents/portfolio_optimization/models.py',
          'meta/env_portfolio_optimization/env_portfolio_optimization.py', 'meta/preprocessor/yahoodownloader.py', 'config_tickers.py']:
    urllib.request.urlretrieve(RAW + f, PKG / pathlib.Path(f).name)
sys.path.insert(0, '/content')
warnings.filterwarnings('ignore'); logging.getLogger('matplotlib.font_manager').disabled = True
import numpy as np, pandas as pd, torch
from sklearn.preprocessing import MaxAbsScaler
from finrl_po.yahoodownloader import YahooDownloader
from finrl_po.env_portfolio_optimization import PortfolioOptimizationEnv
from finrl_po.models import DRLAgent
from finrl_po.architectures import EIIE, EI3
from finrl_po import utils as fu
from finrl_po.config_tickers import DOW_30_TICKER
ROOT = pathlib.Path('/content/drive/MyDrive/졸업프로젝트')
OUT = ROOT / 'data/finrl_replication'; OUT.mkdir(parents=True, exist_ok=True)
MDIR = ROOT / 'models/finrl_replication'; MDIR.mkdir(parents=True, exist_ok=True)
device = 'cpu'
TW, W, H, EPISODES, LR = 50, 60, 60, 40, 0.01
ARCH = {'EIIE': (EIIE, {'k_size': 3, 'time_window': TW}), 'EI3': (EI3, {'time_window': TW})}
FEES, SEEDS = [0.0025, 0.0], [0, 1, 2]
MARKETS = {'BR10': ["VALE3.SA", "PETR4.SA", "ITUB4.SA", "BBDC4.SA", "BBAS3.SA", "RENT3.SA", "LREN3.SA", "PRIO3.SA", "WEGE3.SA", "ABEV3.SA"],
           'DOW30': list(DOW_30_TICKER)}
print('FinRL', COMMIT[:7], '| 다우 종목 수', len(DOW_30_TICKER))

In [ ]:
# 데이터 — 시장별 원자료를 Drive에 저장하고 재사용(재실행 시 같은 데이터)
DATA = {}
for mk, tics in MARKETS.items():
    rp = OUT / f'raw_{mk}.csv'
    if rp.exists():
        raw = pd.read_csv(rp)
    else:
        raw = YahooDownloader(start_date='2011-01-01', end_date='2022-12-31', ticker_list=tics).fetch_data()
        raw.to_csv(rp, index=False)
    raw['date'] = raw['date'].astype(str)
    first = raw.groupby('tic')['date'].min()
    dropped = sorted(first[first > '2011-01-31'].index)
    raw = raw[~raw['tic'].isin(dropped)]
    n_t = raw['tic'].nunique()
    full = raw.groupby('date')['tic'].nunique(); keep = full[full == n_t].index
    raw = raw[raw['date'].isin(keep)].sort_values(['date', 'tic']).reset_index(drop=True)
    num = [c for c in raw.columns if c not in ('date', 'tic', 'day') and pd.api.types.is_numeric_dtype(raw[c])]
    norm = raw.copy()
    for t, g in raw.groupby('tic'):
        norm.loc[g.index, num] = MaxAbsScaler().fit_transform(g[num])
    df = norm[['date', 'tic', 'close', 'high', 'low']].copy()
    dates = sorted(df['date'].unique())
    cw = raw.pivot(index='date', columns='tic', values='close')
    years = {y: df[(df['date'] >= f'{y}-01-01') & (df['date'] < f'{y}-12-31')] for y in (2020, 2021, 2022)}
    DATA[mk] = dict(df=df, dates=dates, close=cw, N=n_t, train=df[(df['date'] >= '2011-01-01') & (df['date'] < '2019-12-31')], years=years)
    missing = sorted(set(tics) - set(raw['tic'].unique()) - set(dropped))
    print(f'{mk}: 종목 {n_t} (상장 늦어 제외 {dropped}, 받지 못함 {missing}) | 거래일 {len(dates)} | 연도별', {y: d.date.nunique() for y, d in years.items()})

In [ ]:
import contextlib, io
# 시작 보유 제어 — FinRL 코드는 그대로, 초기값 두 곳만 교체
_START = ['cash']; _N = [10]
def w0(kind, n):
    return np.array([1] + [0] * n, dtype=np.float32) if kind == 'cash' else np.array([0] + [1 / n] * n, dtype=np.float32)
_ORIG_PVM_RESET = fu.PVM.reset
def _pvm_reset(self):
    _ORIG_PVM_RESET(self)
    self.memory = [w0(_START[0], self.portfolio_size)] * (self.capacity + 1)

class StartEnv(PortfolioOptimizationEnv):
    def _reset_memory(self):
        super()._reset_memory()
        self._actions_memory[0] = w0(_START[0], self.portfolio_size); self._final_weights[0] = w0(_START[0], self.portfolio_size)

def make_env(d, fee, cls=None):
    return (cls or StartEnv)(d, initial_amount=100000, comission_fee_pct=fee, time_window=TW, features=['close', 'high', 'low'], normalize_df=None)

def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)

def new_model(mk, arch, fee):
    cls, kw = ARCH[arch]
    return DRLAgent(make_env(DATA[mk]['train'], fee)).get_model('pg', device, {'lr': LR, 'policy': cls}, dict(kw))

def load_policy(mk, arch, tag):
    cls, kw = ARCH[arch]; p = cls(**kw, device=device); p.load_state_dict(torch.load(MDIR / f'{tag}.pt')); return p

def run(model, policy, d, fee, start, online, seed=0, original=False):
    # FinRL model.test를 그대로 실행. original=True면 패치 없는 원본 클래스로 실행(재현 확인용)
    _START[0] = start; seed_all(seed)
    if original:
        fu.PVM.reset = _ORIG_PVM_RESET; env = make_env(d, fee, PortfolioOptimizationEnv)
    else:
        fu.PVM.reset = _pvm_reset; env = make_env(d, fee)
    with contextlib.redirect_stdout(io.StringIO()):   # FinRL 환경의 종료 시 통계 출력 숨김
        DRLAgent.DRL_validation(model, env, policy=policy, online_training_period=10 if online else 10**9)
    fu.PVM.reset = _pvm_reset; _START[0] = 'cash'
    v = np.asarray(env._asset_memory['final'], dtype=np.float64)
    x = np.asarray([1 - float(a[0]) for a in env._actions_memory[1:]])
    return v, x

fu.PVM.reset = _pvm_reset
CONFIGS = [(a, m, f, s) for a in ARCH for m in MARKETS for f in FEES for s in SEEDS]
tag_of = lambda a, m, f, s: f'{a}_{m}_fee{f}_s{s}'
print('설정', len(CONFIGS), '개')

In [ ]:
%%time
# 학습 — 에피소드마다 체크포인트(정책·옵티마이저·난수 상태). 끝난 설정은 건너뛴다
for a, m, f, s in CONFIGS:
    tag = tag_of(a, m, f, s); final, ck = MDIR / f'{tag}.pt', MDIR / f'{tag}_ckpt.pt'
    if final.exists():
        continue
    seed_all(s); _START[0] = 'cash'
    model = new_model(m, a, f); ep0 = 0
    if ck.exists():
        c = torch.load(ck, weights_only=False)
        model.train_policy.load_state_dict(c['policy']); model.train_optimizer.load_state_dict(c['opt']); ep0 = c['ep']
        random.setstate(c['rng'][0]); np.random.set_state(c['rng'][1]); torch.set_rng_state(c['rng'][2])
        print(f'{tag}: {ep0}에피소드부터 이어서', flush=True)
    for ep in range(ep0, EPISODES):
        model.train(1)
        torch.save({'policy': model.train_policy.state_dict(), 'opt': model.train_optimizer.state_dict(), 'ep': ep + 1,
                    'rng': (random.getstate(), np.random.get_state(), torch.get_rng_state())}, ck)
    torch.save(model.train_policy.state_dict(), final); ck.unlink()
    print('학습 완료:', tag, flush=True)

In [ ]:
def alpha_from_gap(gn):
    # gn[k] = 두 시작(등가중·현금) 경로의 노출 차이 / 시작 차이(=1), k=0이 첫 보유 구간.
    # 기하 감쇠 Σ_k (1−α)^(k+1) 이 관측 누적 차이 Σ gn 과 같아지는 α를 이분법으로 푼다.
    # 관성이 없으면(차이가 바로 0) α→1, 관성이 크면 α→작은 값. 공식이 쓰는 누적 편차와 같은 양이다.
    S, K = float(np.clip(gn, 0, None).sum()), len(gn)
    f = lambda a: ((1 - a) ** (np.arange(K) + 1)).sum() - S
    if S <= 1e-12: return 1.0
    if f(1e-4) < 0: return 1e-4
    lo, hi = 1e-4, 1.0
    for _ in range(100):
        mid = (lo + hi) / 2
        lo, hi = (mid, hi) if f(mid) > 0 else (lo, mid)
    return (lo + hi) / 2


%%time
# 관성 추정(학습 구간) → 예측 확정 → 관측. 설정별로 저장하고, 끝난 설정은 건너뛴다
AP, WP, PP, CP = OUT / 'alpha.csv', OUT / 'windows.csv', OUT / 'protocol.csv', OUT / 'checks.csv'
load = lambda p: pd.read_csv(p).to_dict('records') if p.exists() else []
A, Wr, Pr, Cr = load(AP), load(WP), load(PP), load(CP)
DONE = {r['tag'] for r in A}
Wr, Pr, Cr = [[r for r in X if r['tag'] in DONE] for X in (Wr, Pr, Cr)]

def basket_log(mk, first, n):
    D = DATA[mk]; i = D['dates'].index(first)
    c = D['close'].loc[D['dates'][i:i + n + 1]].to_numpy()
    return np.log1p((c[1:] / c[:-1] - 1).mean(axis=1))

def span(mk, first, end, before):
    D = DATA[mk]; p = D['dates'].index(first)
    return D['df'][(D['df']['date'] >= D['dates'][p - (TW - 1) - before]) & (D['df']['date'] <= end)]

BENCH = {}
for a, m, f, s in CONFIGS:
    tag = tag_of(a, m, f, s)
    if tag in DONE:
        continue
    D = DATA[m]; pol = load_policy(m, a, tag); model = new_model(m, a, f)
    trd = sorted(D['train']['date'].unique()); gaps, xs = [], []
    for p in np.linspace(TW - 1 + W, len(trd) - H - 2, 20).astype(int):
        on = span(m, trd[p], trd[p + H], 0)
        _, xe = run(model, pol, on, f, 'ew', False); _, xc = run(model, pol, on, f, 'cash', False)
        _, xw = run(model, pol, span(m, trd[p], trd[p + H], W), f, 'cash', False)
        k = min(len(xe), len(xc), H); gaps.append(np.abs(xe[:k] - xc[:k])); xs.append(xw[W:].mean())
    L = min(map(len, gaps)); g = np.mean([q[:L] for q in gaps], axis=0)
    a_eff, xbar = alpha_from_gap(g), float(np.mean(xs))
    rw, rp, rc = [], [], []
    yd = {y: sorted(d['date'].unique()) for y, d in D['years'].items()}
    pred = {}
    for y, dl in yd.items():   # 예측을 먼저 확정
        bl = basket_log(m, dl[TW - 1], len(dl) - TW)
        for st, x0 in (('cash', 0.0), ('ew', 1.0)):
            pred[(y, st)] = float((x0 - xbar) * ((1 - a_eff) ** (np.arange(len(bl)) + 1) * bl).sum())
    for y, dl in yd.items():
        dy = D['years'][y]
        vw, _ = run(model, pol, span(m, dl[TW - 1], dl[-1], W), f, 'cash', False); gw = np.log(vw[-1] / vw[W])
        for st in ('cash', 'ew'):
            v, _ = run(model, pol, dy, f, st, False)
            rw.append({'tag': tag, '구조': a, '시장': m, '수수료': f, 'seed': s, '연도': y, '시작': st, '예측': pred[(y, st)], '관측': float(np.log(v[-1] / v[0]) - gw)})
            vo, _ = run(model, pol, dy, f, st, True, seed=s)
            rp.append({'tag': tag, '구조': a, '시장': m, '수수료': f, 'seed': s, '연도': y, '시작': st, 'log 성장': float(np.log(vo[-1] / vo[0]))})
            if st == 'cash' and y == 2020:   # 재현 확인: 패치 없는 원본과 같은 값인가
                vo2, _ = run(model, pol, dy, f, 'cash', True, seed=s, original=True)
                rc.append({'tag': tag, '패치 현금 시작': float(vo[-1]), '원본': float(vo2[-1]), '일치': bool(np.isclose(vo[-1], vo2[-1], rtol=1e-9))})
        if (m, f, y) not in BENCH:   # 데모 벤치마크(매 스텝 등가중, 현금 시작)
            env = make_env(dy, f, PortfolioOptimizationEnv); env.reset(); term = False
            with contextlib.redirect_stdout(io.StringIO()):
                while not term:
                    _, _, term, _ = env.step([0] + [1 / D['N']] * D['N'])
            BENCH[(m, f, y)] = float(np.log(env._asset_memory['final'][-1] / env._asset_memory['final'][0]))
        rp.append({'tag': tag, '구조': a, '시장': m, '수수료': f, 'seed': s, '연도': y, '시작': '벤치마크', 'log 성장': BENCH[(m, f, y)]})
    Wr += rw; Pr += rp; Cr += rc
    for X, P_ in ((Wr, WP), (Pr, PP), (Cr, CP)):
        pd.DataFrame(X).to_csv(P_, index=False)
    A.append({'tag': tag, '구조': a, '시장': m, '수수료': f, 'seed': s, 'α_eff': a_eff, 'x̄': xbar, '초기 노출 차이': float(g[0])})
    pd.DataFrame(A).to_csv(AP, index=False)   # 완료 표시는 마지막에
    print(f'{tag}: α_eff={a_eff:.4f} x̄={xbar:.3f} | 재현 일치 {rc[0]["일치"]}', flush=True)
print('평가 완료')

In [ ]:
# 사전 등록 판정
A_, W_, P_, C_ = (pd.DataFrame(X) for X in (A, Wr, Pr, Cr))
print('재현 확인(패치 현금 시작 = 원본):', int(C_['일치'].sum()), '/', len(C_))
print(A_.pivot_table(index=['구조', '시장', 'seed'], columns='수수료', values='α_eff').round(4).to_string())
cells = A_[A_['수수료'] == 0.0025].groupby(['구조', '시장'])['α_eff'].apply(lambda v: int((v < 0.5).sum()))
h6 = int((cells >= 2).sum())
print(f"\n[H6] (구조×시장) 칸별 α_eff<0.5 시드 수: {cells.to_dict()} → {h6}/4 칸 충족 → {'지지' if h6 >= 3 else '지지 안 됨'}")
pv = A_.pivot_table(index=['구조', '시장', 'seed'], columns='수수료', values='α_eff')
h6b = int((pv[0.0025] < pv[0.0]).sum())
print(f"[H6b] α_eff(0.25%) < α_eff(0%): {h6b}/{len(pv)} → {'지지' if h6b >= 9 else '지지 안 됨'}")
if W_['예측'].std() * 100 < 0.5:
    print(f"[H7] 판정 불가 — 예측 표준편차 {W_['예측'].std()*100:.2f} 로그%p < 0.5(예측 범위가 좁아 상관이 무의미). 관측 |효과| 최대 {W_['관측'].abs().max()*100:.2f}, 95분위 {W_['관측'].abs().quantile(0.95)*100:.2f} 로그%p")
else:
    r = np.corrcoef(W_['예측'], W_['관측'])[0, 1]; sl = np.polyfit(W_['예측'], W_['관측'], 1)[0]
    print(f"[H7] 쌍 {len(W_)}개: 상관 {r:.3f}, 기울기 {sl:.2f} → {'지지' if r >= 0.8 and 0.7 <= sl <= 1.3 else '지지 안 됨'}")
for (a, m), g in W_.groupby(['구조', '시장']):
    print(f"   {a} {m}: 상관 {np.corrcoef(g['예측'], g['관측'])[0, 1]:.3f}, 평균|관측| {g['관측'].abs().mean()*100:.2f} 로그%p")
Q = P_[P_['수수료'] == 0.0025].pivot_table(index=['구조', '시장', 'seed', '연도'], columns='시작', values='log 성장')
Q['Δ'] = (Q['ew'] - Q['cash']) * 100
Q['초과(현금)'] = (Q['cash'] - Q['벤치마크']) * 100; Q['초과(등가중)'] = (Q['ew'] - Q['벤치마크']) * 100
Q['부호 바뀜'] = np.sign(Q['초과(현금)']) != np.sign(Q['초과(등가중)'])
med = Q.groupby(['구조', '시장'])['Δ'].apply(lambda v: v.abs().median())
h8 = int((med >= 1.0).sum())
print(Q.round(2).to_string())
print(f"\n[H8] 칸별 |Δ| 중앙값(로그%p): {med.round(2).to_dict()} → {h8}/4 칸 ≥1.0 → {'실질적 영향' if h8 >= 3 else '실질적 영향 아님'} | 초과수익 부호 바뀜 {int(Q['부호 바뀜'].sum())}/{len(Q)}")
Z = P_[P_['수수료'] == 0.0].pivot_table(index=['구조', '시장', 'seed', '연도'], columns='시작', values='log 성장')
print(f"[대조군 수수료 0%] |Δ| 중앙값 {((Z['ew'] - Z['cash']).abs() * 100).median():.2f} 로그%p")